# Tempo de CPU e cardinalidade final

A figura resume as 10 repetições de cada combinação cenário–método. O ponto representa a mediana e a barra vertical representa o intervalo interquartil ($Q_1$–$Q_3$). Os dois eixos verticais usam escala logarítmica. O NBI original é mostrado somente nos cenários com $m=4$.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
METRICS_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_metrics.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'computational_cost_cardinality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIO_ORDER = [
    'm4_low', 'm4_medium', 'm4_high',
    'm6_low', 'm6_medium', 'm6_high',
    'm12_low', 'm12_medium', 'm12_high',
]
METHOD_ORDER = ['CNBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D', 'NBI']
METHOD_LABEL = {
    'CNBI': 'C-NBI', 'VRF-NBI': 'VRF-NBI', 'NSGA-III': 'NSGA-III',
    'MOEA/D': 'MOEA/D', 'NBI': 'NBI original',
}
METHOD_COLOR = {
    'CNBI': '#E69F00', 'VRF-NBI': '#0072B2', 'NSGA-III': '#009E73',
    'MOEA/D': '#D62728', 'NBI': '#7B3294',
}
METHOD_MARKER = {'CNBI': 'o', 'VRF-NBI': '^', 'NSGA-III': 's', 'MOEA/D': 'D', 'NBI': 'P'}
METHOD_OFFSET = {'CNBI': -0.18, 'VRF-NBI': -0.09, 'NSGA-III': 0.0, 'MOEA/D': 0.09, 'NBI': 0.18}

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 8.5,
    'axes.titlesize': 10.0,
    'axes.labelsize': 9.0,
    'xtick.labelsize': 7.7,
    'ytick.labelsize': 7.7,
    'legend.fontsize': 7.8,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Fonte:', METRICS_PATH.relative_to(ROOT))

In [ ]:
metrics = pd.read_csv(METRICS_PATH)
complete = metrics.loc[metrics['comparison'].eq('complete')].copy()
complete = complete.loc[
    complete['scenario'].isin(SCENARIO_ORDER)
    & complete['method'].isin(METHOD_ORDER)
].copy()
assert complete[['cpu_seconds', 'n']].gt(0).all().all()

summary = (
    complete.groupby(['scenario', 'method'], sort=False)
    .agg(
        repetitions=('seed', 'nunique'),
        cpu_q1=('cpu_seconds', lambda values: values.quantile(0.25)),
        cpu_median=('cpu_seconds', 'median'),
        cpu_q3=('cpu_seconds', lambda values: values.quantile(0.75)),
        cardinality_q1=('n', lambda values: values.quantile(0.25)),
        cardinality_median=('n', 'median'),
        cardinality_q3=('n', lambda values: values.quantile(0.75)),
    )
    .reset_index()
)
summary['scenario'] = pd.Categorical(summary['scenario'], SCENARIO_ORDER, ordered=True)
summary['method'] = pd.Categorical(summary['method'], METHOD_ORDER, ordered=True)
summary = summary.sort_values(['scenario', 'method']).reset_index(drop=True)
assert len(summary) == 39
assert summary['repetitions'].eq(10).all()
assert summary.loc[summary['method'].eq('NBI'), 'scenario'].astype(str).str.startswith('m4_').all()
summary_path = OUT_DIR / 'full_cpu_cardinality_median_iqr.csv'
summary.to_csv(summary_path, index=False)
summary.head()

In [ ]:
PANEL_SPECS = [
    ('cpu', '(a) Tempo de CPU', 'Tempo de CPU (s)'),
    ('cardinality', '(b) Cardinalidade final', 'Cardinalidade final'),
]
x_base = np.arange(len(SCENARIO_ORDER), dtype=float)
x_lookup = dict(zip(SCENARIO_ORDER, x_base))

figure, axes = plt.subplots(
    2, 1, sharex=True,
    figsize=(16 / 2.54, 14.2 / 2.54),
)
figure.subplots_adjust(left=0.105, right=0.99, top=0.965, bottom=0.22, hspace=0.20)

for axis, (prefix, title, ylabel) in zip(axes, PANEL_SPECS):
    for method in METHOD_ORDER:
        block = summary.loc[summary['method'].eq(method)].copy()
        if block.empty:
            continue
        x = np.array([x_lookup[str(scenario)] for scenario in block['scenario']], dtype=float)
        x = x + METHOD_OFFSET[method]
        median = block[f'{prefix}_median'].to_numpy(dtype=float)
        q1 = block[f'{prefix}_q1'].to_numpy(dtype=float)
        q3 = block[f'{prefix}_q3'].to_numpy(dtype=float)
        errors = np.vstack([median - q1, q3 - median])
        axis.plot(
            x, median, color=METHOD_COLOR[method], linewidth=0.9, alpha=0.80,
            marker=METHOD_MARKER[method], markersize=4.8 if method == 'CNBI' else 4.2,
            markerfacecolor=METHOD_COLOR[method], markeredgecolor='white',
            markeredgewidth=0.35, zorder=3,
        )
        axis.errorbar(
            x, median, yerr=errors, fmt='none', color=METHOD_COLOR[method],
            elinewidth=0.9, capsize=2.2, capthick=0.8, alpha=0.95, zorder=2,
        )

    axis.set_yscale('log')
    axis.set_ylabel(ylabel)
    axis.set_title(title, loc='left', pad=4)
    axis.set_xlim(-0.48, len(SCENARIO_ORDER) - 0.52)
    axis.grid(axis='y', which='major', color='0.72', linewidth=0.55, alpha=0.55)
    axis.grid(axis='y', which='minor', color='0.84', linewidth=0.35, alpha=0.35)
    axis.set_axisbelow(True)
    for separator in (2.5, 5.5):
        axis.axvline(separator, color='0.55', linewidth=0.7, zorder=0)
    for spine in axis.spines.values():
        spine.set_color('0.35')
        spine.set_linewidth(0.65)

axes[-1].set_xticks(x_base, SCENARIO_ORDER, rotation=35, ha='right', rotation_mode='anchor')
axes[-1].set_xlabel('Cenário')

legend_handles = [
    Line2D(
        [0], [0], color=METHOD_COLOR[method], marker=METHOD_MARKER[method],
        linewidth=0.9, markersize=4.8 if method == 'CNBI' else 4.2,
        markerfacecolor=METHOD_COLOR[method], markeredgecolor='white',
        markeredgewidth=0.35, label=METHOD_LABEL[method],
    )
    for method in METHOD_ORDER
]
figure.legend(
    handles=legend_handles, loc='lower center', bbox_to_anchor=(0.55, 0.025),
    ncol=5, frameon=False, columnspacing=1.25, handletextpad=0.45,
)

png_path = OUT_DIR / 'fig_tempo_cpu_cardinalidade_mediana_iqr.png'
pdf_path = OUT_DIR / 'fig_tempo_cpu_cardinalidade_mediana_iqr.pdf'
figure.savefig(png_path, dpi=300, bbox_inches='tight', pad_inches=0.02)
figure.savefig(pdf_path, bbox_inches='tight', pad_inches=0.02)
plt.close(figure)
print('Figura:', png_path.relative_to(ROOT))

In [ ]:
metadata = {
    'source': METRICS_PATH.relative_to(ROOT).as_posix(),
    'comparison': 'complete',
    'statistics': 'median and interquartile interval Q1-Q3 over 10 repetitions',
    'cpu_variable': 'cpu_seconds',
    'cardinality_variable': 'n',
    'scenarios': SCENARIO_ORDER,
    'methods': [METHOD_LABEL[method] for method in METHOD_ORDER],
    'nbi_original_rule': 'shown only for m=4 scenarios',
    'y_scales': {'cpu': 'logarithmic', 'cardinality': 'logarithmic'},
    'method_colors': {METHOD_LABEL[key]: value for key, value in METHOD_COLOR.items()},
    'publication_width_cm': 16,
}
metadata_path = OUT_DIR / 'tempo_cpu_cardinalidade_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
for artifact in [summary_path, png_path, pdf_path, metadata_path]:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Combinações cenário–método:', len(summary))